In [8]:
# ================================================
# 1. Imports
# ================================================
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

In [9]:
# ================================================
# 2. Load datasets (train + eval)
# ================================================
train_df = pd.read_csv("/Users/momo/MLOps/Regression_Model/data/processed/feature_engineered_train.csv")
eval_df  = pd.read_csv("/Users/momo/MLOps/Regression_Model/data/processed/feature_engineered_eval.csv")

In [10]:
# ================================================
# 3. Drop leakage & fully-null columns
# ================================================

# median_sale_price has the highest correlation with price => data leakage
# lat / lng are 100% NaN in this dataset => would break StandardScaler
cols_to_drop = ["median_sale_price", "lat", "lng"]
cols_to_drop = [c for c in cols_to_drop if c in train_df.columns]

train_df.drop(columns=cols_to_drop, inplace=True)
eval_df.drop(columns=cols_to_drop, inplace=True)

print("Dropped:", cols_to_drop)
print("Remaining features:", train_df.shape[1] - 1)  # -1 for target

Dropped: ['lat', 'lng']
Remaining features: 37


In [11]:
# ================================================
# 4. Define target & features
# ================================================
target = "price"
X_train = train_df.drop(columns=[target])
y_train = train_df[target]

X_eval = eval_df.drop(columns=[target])
y_eval = eval_df[target]

In [12]:
# ================================================
# 5. Standardization (fit on train, transform eval)
# ================================================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_eval_scaled  = scaler.transform(X_eval)

In [13]:
# ================================================
# 6. Train & Evaluate Models
# ================================================

# --- Linear Regression ---
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_eval_scaled)

print("Linear Regression:")
print(" MAE:", mean_absolute_error(y_eval, y_pred_lr))
print(" RMSE:", np.sqrt(mean_squared_error(y_eval, y_pred_lr)))
print(" R²:", r2_score(y_eval, y_pred_lr))

Linear Regression:
 MAE: 53811.93813401003
 RMSE: 121336.13469296062
 R²: 0.8862267031700102


In [14]:
# --- Ridge Regression ---
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)
y_pred_ridge = ridge.predict(X_eval_scaled)

print("\nRidge Regression:")
print(" MAE:", mean_absolute_error(y_eval, y_pred_ridge))
print(" RMSE:", np.sqrt(mean_squared_error(y_eval, y_pred_ridge)))
print(" R²:", r2_score(y_eval, y_pred_ridge))


Ridge Regression:
 MAE: 53811.11466043951
 RMSE: 121338.02551498688
 R²: 0.8862231572068496


In [15]:
# --- Lasso Regression ---
lasso = Lasso(alpha=0.1, max_iter=10000)
lasso.fit(X_train_scaled, y_train)
y_pred_lasso = lasso.predict(X_eval_scaled)

print("\nLasso Regression:")
print(" MAE:", mean_absolute_error(y_eval, y_pred_lasso))
print(" RMSE:", np.sqrt(mean_squared_error(y_eval, y_pred_lasso)))
print(" R²:", r2_score(y_eval, y_pred_lasso))


Lasso Regression:
 MAE: 53845.2203214825
 RMSE: 121376.73759927296
 R²: 0.8861505461464898


/Users/momo/MLOps/Regression_Model/.venv/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.083e+15, tolerance: 5.209e+12
  model = cd_fast.enet_coordinate_descent(


In [16]:
# --- ElasticNet ---
elastic = ElasticNet(alpha=0.1, l1_ratio=0.5)
elastic.fit(X_train_scaled, y_train)
y_pred_elastic = elastic.predict(X_eval_scaled)

print("\nElasticNet Regression:")
print(" MAE:", mean_absolute_error(y_eval, y_pred_elastic))
print(" RMSE:", np.sqrt(mean_squared_error(y_eval, y_pred_elastic)))
print(" R²:", r2_score(y_eval, y_pred_elastic))


ElasticNet Regression:
 MAE: 54234.249320614595
 RMSE: 122295.84870428537
 R²: 0.8844197946433394


In [17]:
# ================================================
# 7. Model Comparison Summary
# ================================================
results = {
    "Linear Regression": (y_pred_lr,   lr),
    "Ridge (α=1.0)":     (y_pred_ridge, ridge),
    "Lasso (α=0.1)":     (y_pred_lasso, lasso),
    "ElasticNet (α=0.1, l1=0.5)": (y_pred_elastic, elastic),
}

print(f"{'Model':<30} {'MAE':>12} {'RMSE':>12} {'R²':>8}")
print("-" * 66)
for name, (y_pred, _) in results.items():
    mae  = mean_absolute_error(y_eval, y_pred)
    rmse = np.sqrt(mean_squared_error(y_eval, y_pred))
    r2   = r2_score(y_eval, y_pred)
    print(f"{name:<30} {mae:>12,.0f} {rmse:>12,.0f} {r2:>8.4f}")

Model                                   MAE         RMSE       R²
------------------------------------------------------------------
Linear Regression                    53,812      121,336   0.8862
Ridge (α=1.0)                        53,811      121,338   0.8862
Lasso (α=0.1)                        53,845      121,377   0.8862
ElasticNet (α=0.1, l1=0.5)           54,234      122,296   0.8844
